# Targeted Fixing Tool (Gold Labeling)

Use this notebook to manually fix 'Unknown', 'Unsure', or 'Regex fallback' labels.
Any label you fix here becomes **Gold** status.

In [9]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# --- Configuration ---
LABELS_PATH = '../data/labels/action_labels_llm_clean_refined.csv'

# Action Taxonomy
TAXONOMY = [
    "Locomotion",
    "Essential Operation",
    "Object Transfer",
    "Search",
    "Error / Correction",
    "Stationary",
    "Unknown" 
]

In [10]:
# --- Helper Functions ---

def load_data():
    if not os.path.exists(LABELS_PATH):
        print("File not found!")
        return pd.DataFrame()
    return pd.read_csv(LABELS_PATH)

def save_row(df, index, new_action, status='gold'):
    df.at[index, 'action'] = new_action
    df.at[index, 'status'] = status
    df.to_csv(LABELS_PATH, index=False)
    print(f"Saved row {index} as {new_action} ({status})")

def delete_row(df, index):
    # We mark as 'deleted' instead of dropping to preserve indexing for this session
    df.at[index, 'status'] = 'deleted'
    df.to_csv(LABELS_PATH, index=False)
    print(f"Marked row {index} as DELETED")

In [11]:
# --- Load & Filter Data ---
df = load_data()

# Filter for items needing repair
# Criteria: Unknown OR Regex fallback OR Error
to_fix_mask = (
    (df['action'] == 'Unknown') | 
    (df['action'].astype(str).str.contains('Error', case=False, na=False)) | 
    (df['reasoning'].astype(str).str.contains('fallback', case=False, na=False)) | 
    (df['reasoning'].astype(str).str.contains('Error', case=False, na=False))
) & (df['status'] != 'gold') & (df['status'] != 'deleted')

fix_indices = df[to_fix_mask].index.tolist()
print(f"Found {len(fix_indices)} rows to fix.")

Found 17997 rows to fix.


In [12]:
# --- Interactive Tool ---

# 1. Select Batch Size
batch_size_input = widgets.IntText(value=50, description='Batch Size:')
display(batch_size_input)

current_idx_pointer = 0
indices_to_process = []

# --- Display Components ---
info_box = widgets.HTML()

# Controls
dropdown = widgets.Dropdown(description='Label:', options=TAXONOMY)
save_btn = widgets.Button(description="SAVE (Gold)", button_style='success')
delete_btn = widgets.Button(description="DELETE", button_style='danger')
skip_btn = widgets.Button(description="Skip", button_style='')

control_box = widgets.HBox([dropdown, save_btn, delete_btn, skip_btn])
main_layout = widgets.VBox([info_box, control_box])

# Initially hide until started
main_layout.layout.display = 'none'

display(main_layout)

def start_labeling(b):
    global indices_to_process, current_idx_pointer
    limit = batch_size_input.value
    indices_to_process = fix_indices[:limit]
    current_idx_pointer = 0
    main_layout.layout.display = 'flex'
    show_sample()

def show_sample():
    global current_idx_pointer
    
    if current_idx_pointer >= len(indices_to_process):
        info_box.value = "<h3>Batch completed! Reload data or restart to fix more.</h3>"
        control_box.layout.display = 'none' # Hide controls
        return
    
    # Make sure controls are visible
    control_box.layout.display = 'flex'
        
    row_idx = indices_to_process[current_idx_pointer]
    row = df.loc[row_idx]
    
    # Update Info Box with HTML
    # Using styling to separate from controls
    info_html = f"""
    <div style="border:1px solid #ddd; padding:10px; margin-bottom:10px; background-color:#f9f9f9;">
        <div style="margin-bottom:10px; border-bottom:1px solid #ccc; padding-bottom:5px;">
            <strong>Batch Progress:</strong> {current_idx_pointer+1}/{len(indices_to_process)} 
            <span style="float:right; color:#666;"><strong>Total Pool:</strong> {len(fix_indices)} rows</span>
        </div>
        <p><strong>Row ID:</strong> {row_idx} | <strong>Time:</strong> {row['timestamp_sec']}s</p>
        <p><strong>Context:</strong> {row['scenario']}</p>
        <p><strong>Narration:</strong> <span style="color:blue; font-size:1.1em;">{row['narration_text']}</span></p>
        <p><strong>Current Label:</strong> {row['action']} <em>({row['status']})</em></p>
        <p><strong>Reasoning:</strong> {row['reasoning']}</p>
    </div>
    """
    info_box.value = info_html
    
    # Update Dropdown
    dropdown.value = row['action'] if row['action'] in TAXONOMY else 'Unknown'

def on_save(b):
    global current_idx_pointer
    if current_idx_pointer < len(indices_to_process):
        row_idx = indices_to_process[current_idx_pointer]
        save_row(df, row_idx, dropdown.value, status='gold')
        current_idx_pointer += 1
        show_sample()
    
def on_delete(b):
    global current_idx_pointer
    if current_idx_pointer < len(indices_to_process):
        row_idx = indices_to_process[current_idx_pointer]
        delete_row(df, row_idx)
        current_idx_pointer += 1
        show_sample()
    
def on_skip(b):
    global current_idx_pointer
    if current_idx_pointer < len(indices_to_process):
        current_idx_pointer += 1
        show_sample()
    
save_btn.on_click(on_save)
delete_btn.on_click(on_delete)
skip_btn.on_click(on_skip)

start_btn = widgets.Button(description="Start Batch")
start_btn.on_click(start_labeling)
display(start_btn)

IntText(value=50, description='Batch Size:')

Button(description='Start Batch', style=ButtonStyle())